In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

In [3]:
!pip install duckdb

In [4]:
df=pd.read_csv("Cleaned_yellow_tripdata_2025-01.csv")

In [5]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,extra,...,improvement_surcharge,total_amount,cbd_congestion_fee,trip duration min,pickup hour,pickup day,pickup_weekday,revenue_per_mile,revenue_per_minute,tip_percentage
0,2,2026-01-01 00:01:27,2026-01-01 00:44:30,1.0,18.47,132,246,1,70.0,0.00,...,1.0,99.78,0.75,43.05,0,1,3,5.40,2.32,0.23
1,2,2026-01-01 00:51:26,2026-01-01 01:17:35,1.0,5.89,170,37,1,30.3,1.00,...,1.0,51.59,0.75,26.15,0,1,3,8.76,1.97,0.28
2,1,2026-01-01 00:32:16,2026-01-01 01:47:51,1.0,11.10,137,133,1,64.6,4.25,...,1.0,92.74,0.75,75.58,0,1,3,8.35,1.23,0.24
3,2,2026-01-01 00:00:22,2026-01-01 00:27:41,2.0,19.15,132,75,2,70.0,0.00,...,1.0,80.19,0.00,27.32,0,1,3,4.19,2.94,0.00
4,2,2026-01-01 00:31:18,2026-01-01 00:50:00,4.0,11.49,48,265,1,135.0,0.00,...,1.0,150.81,0.75,18.70,0,1,3,13.13,8.06,0.00


In [6]:
df.shape

(231954, 23)

In [7]:
hourly_df = (
    df.groupby("pickup hour")
      .agg(
          total_trips=("VendorID", "count"),
          total_revenue=("total_amount", "sum"),
          avg_fare=("fare_amount", "mean"),
          avg_tip=("tip_amount", "mean"),
          avg_trip_duration=("trip duration min", "mean")
      )
      .reset_index()
)

In [8]:
hourly_df

,pickup hour,total_trips,total_revenue,avg_fare,avg_tip,avg_trip_duration
0,0,6672,472278.06,49.815772,6.402259,27.286196
1,1,3652,240863.10,48.051309,4.199789,26.320235
2,2,2243,140336.96,45.870584,3.278511,25.574971
3,3,2016,130194.78,47.526493,3.674519,25.915174
4,4,2480,169883.25,50.033879,5.200722,28.419194
5,5,3802,281711.26,54.460487,6.488567,34.826444
6,6,6248,470878.85,56.144646,5.989125,40.262910
7,7,8461,647989.52,56.356153,6.413670,43.981061
8,8,8988,691872.16,56.101474,6.944551,44.340714
9,9,9534,730120.57,54.347003,7.660288,41.199155


In [11]:
hourly_df["revenue_per_trip"] = hourly_df["total_revenue"] / hourly_df["total_trips"]


In [13]:
hourly_df["demand_percentile"] = round(hourly_df["total_trips"].rank(pct=True) * 100, 2)
hourly_df

,pickup hour,total_trips,total_revenue,avg_fare,avg_tip,avg_trip_duration,revenue_per_trip,demand_percentile
0,0,6672,472278.06,49.815772,6.402259,27.286196,70.785081,29.17
1,1,3652,240863.10,48.051309,4.199789,26.320235,65.953751,16.67
2,2,2243,140336.96,45.870584,3.278511,25.574971,62.566634,8.33
3,3,2016,130194.78,47.526493,3.674519,25.915174,64.580744,4.17
4,4,2480,169883.25,50.033879,5.200722,28.419194,68.501310,12.50
5,5,3802,281711.26,54.460487,6.488567,34.826444,74.095544,20.83
6,6,6248,470878.85,56.144646,5.989125,40.262910,75.364733,25.00
7,7,8461,647989.52,56.356153,6.413670,43.981061,76.585453,33.33
8,8,8988,691872.16,56.101474,6.944551,44.340714,76.977321,37.50
9,9,9534,730120.57,54.347003,7.660288,41.199155,76.580718,41.67


In [14]:
def surge_multiplier(p):
    if p < 75:
        return 1.0
    elif p < 85:
        return 1.2
    elif p < 95:
        return 1.5
    else:
        return 2.0

hourly_df["surge_multiplier"] = hourly_df["demand_percentile"].apply(surge_multiplier)

hourly_df

,pickup hour,total_trips,total_revenue,avg_fare,avg_tip,avg_trip_duration,revenue_per_trip,demand_percentile,surge_multiplier
0,0,6672,472278.06,49.815772,6.402259,27.286196,70.785081,29.17,1.0
1,1,3652,240863.10,48.051309,4.199789,26.320235,65.953751,16.67,1.0
2,2,2243,140336.96,45.870584,3.278511,25.574971,62.566634,8.33,1.0
3,3,2016,130194.78,47.526493,3.674519,25.915174,64.580744,4.17,1.0
4,4,2480,169883.25,50.033879,5.200722,28.419194,68.501310,12.50,1.0
5,5,3802,281711.26,54.460487,6.488567,34.826444,74.095544,20.83,1.0
6,6,6248,470878.85,56.144646,5.989125,40.262910,75.364733,25.00,1.0
7,7,8461,647989.52,56.356153,6.413670,43.981061,76.585453,33.33,1.0
8,8,8988,691872.16,56.101474,6.944551,44.340714,76.977321,37.50,1.0
9,9,9534,730120.57,54.347003,7.660288,41.199155,76.580718,41.67,1.0


In [15]:
hourly_df["surge_revenue"] = hourly_df["total_revenue"] *hourly_df["surge_multiplier"]
hourly_df

,pickup hour,total_trips,total_revenue,avg_fare,avg_tip,avg_trip_duration,revenue_per_trip,demand_percentile,surge_multiplier,surge_revenue
0,0,6672,472278.06,49.815772,6.402259,27.286196,70.785081,29.17,1.0,472278.060
1,1,3652,240863.10,48.051309,4.199789,26.320235,65.953751,16.67,1.0,240863.100
2,2,2243,140336.96,45.870584,3.278511,25.574971,62.566634,8.33,1.0,140336.960
3,3,2016,130194.78,47.526493,3.674519,25.915174,64.580744,4.17,1.0,130194.780
4,4,2480,169883.25,50.033879,5.200722,28.419194,68.501310,12.50,1.0,169883.250
5,5,3802,281711.26,54.460487,6.488567,34.826444,74.095544,20.83,1.0,281711.260
6,6,6248,470878.85,56.144646,5.989125,40.262910,75.364733,25.00,1.0,470878.850
7,7,8461,647989.52,56.356153,6.413670,43.981061,76.585453,33.33,1.0,647989.520
8,8,8988,691872.16,56.101474,6.944551,44.340714,76.977321,37.50,1.0,691872.160
9,9,9534,730120.57,54.347003,7.660288,41.199155,76.580718,41.67,1.0,730120.570


In [19]:
final_df = df.copy()

# Merging surge multiplier by hour
final_df = final_df.merge(
    hourly_df[["pickup hour", "surge_multiplier"]],
    on="pickup hour",
    how="left"
)

# Calculating simulated surge fare
final_df["simulated_total_amount"] = (
    final_df["total_amount"] *
    final_df["surge_multiplier"]
)

final_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,extra,...,cbd_congestion_fee,trip duration min,pickup hour,pickup day,pickup_weekday,revenue_per_mile,revenue_per_minute,tip_percentage,surge_multiplier,simulated_total_amount
0,2,2026-01-01 00:01:27,2026-01-01 00:44:30,1.0,18.47,132,246,1,70.0,0.00,...,0.75,43.05,0,1,3,5.40,2.32,0.23,1.0,99.78
1,2,2026-01-01 00:51:26,2026-01-01 01:17:35,1.0,5.89,170,37,1,30.3,1.00,...,0.75,26.15,0,1,3,8.76,1.97,0.28,1.0,51.59
2,1,2026-01-01 00:32:16,2026-01-01 01:47:51,1.0,11.10,137,133,1,64.6,4.25,...,0.75,75.58,0,1,3,8.35,1.23,0.24,1.0,92.74
3,2,2026-01-01 00:00:22,2026-01-01 00:27:41,2.0,19.15,132,75,2,70.0,0.00,...,0.00,27.32,0,1,3,4.19,2.94,0.00,1.0,80.19
4,2,2026-01-01 00:31:18,2026-01-01 00:50:00,4.0,11.49,48,265,1,135.0,0.00,...,0.75,18.70,0,1,3,13.13,8.06,0.00,1.0,150.81


In [20]:
final_df.to_csv("Cleaned_yellow_tripdata_2025-011.csv",index=False)